# UPGPR on MOOCCube (Xuetang) - Colab

Runs the full pipeline from `README.md` on a Colab GPU runtime:
preprocess -> make_dataset -> train_transe -> train_agent -> test_agent.

- Repo is cloned from upstream into the runtime (`/content/courserec`). Local edits are not synced.
- Deps live in a Python 3.10 venv (repo pins 3.10), so Colab's own packages are untouched.
- Outputs (`tmp/`) go to Google Drive if it mounts, otherwise to `/content/persist` (lost when the runtime ends).
- Each heavy step is skipped if its marker file (`tmp/mooc/.done_<step>[_e<epochs>]`) exists. Markers include the epoch count, so smoke runs never block full runs. Delete a marker to force a rerun.

In [ ]:
import os, sys, json, subprocess, zipfile, shutil
from pathlib import Path

REPO_URL = "https://github.com/epfl-ml4ed/courserec.git"
REPO_DIR = Path("/content/courserec")
VENV = Path("/content/venv")
PY = str(VENV / "bin" / "python")
MOOCCUBE_URL = "http://lfs.aminer.cn/misc/moocdata/data/MOOCCube.zip"
CONFIG = "config/UPGPR/mooc.json"

# Set both to a small number (e.g. 2) for a quick smoke test. None = use values from the config.
EPOCHS_EMBED = None
EPOCHS_AGENT = None

os.environ["WANDB_MODE"] = "disabled"
os.environ["PYTHONUNBUFFERED"] = "1"


def sh(cmd, cwd=None, check=True):
    """Run a shell command, stream output live, raise on non-zero exit."""
    print(f"$ {cmd}")
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, bufsize=0)
    while True:
        chunk = proc.stdout.read(4096)
        if not chunk:
            break
        print(chunk.decode(errors="replace"), end="")
    code = proc.wait()
    if check and code != 0:
        raise RuntimeError(f"command failed with exit code {code}: {cmd}")
    return code


sh("nvidia-smi", check=False)

## 1. Persistent storage

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST = Path("/content/drive/MyDrive/courserec")
except Exception as e:
    print("Drive not available, using local storage (lost when the runtime ends):", repr(e))
    PERSIST = Path("/content/persist")
PERSIST.mkdir(parents=True, exist_ok=True)
print("PERSIST =", PERSIST)

## 2. Clone repo

In [ ]:
if (REPO_DIR / ".git").exists():
    sh(f"git -C {REPO_DIR} pull --ff-only")
else:
    sh(f"git clone {REPO_URL} {REPO_DIR}")
os.chdir(REPO_DIR)
print("cwd =", os.getcwd())

## 3. Dependencies (Python 3.10 venv)

In [ ]:
sh("pip install -q uv")
if not Path(PY).exists():
    sh(f"uv venv --python 3.10 {VENV}")
# setuptools<81: wandb 0.16.3 needs pkg_resources
sh(f'uv pip install -q --python {PY} -r requirements.txt "setuptools<81"')
sh(f'{PY} -c "import torch, recbole, skillNer, wandb, easydict; '
   f'print(\'torch\', torch.__version__, \'cuda:\', torch.cuda.is_available())"')

## 4. MOOCCube data

In [ ]:
NEEDED = {"course-concept.json", "school-course.json", "teacher-course.json", "user-course.json"}
rel_dst = REPO_DIR / "data/mooc/MOOCCube/relations"

if all((rel_dst / n).exists() for n in NEEDED):
    print("relations already in place")
else:
    cache = PERSIST / "cache"
    cache.mkdir(exist_ok=True)
    zip_path = cache / "MOOCCube.zip"
    if not zip_path.exists():
        part = cache / "MOOCCube.zip.part"
        sh(f"wget -c -nv -O {part} {MOOCCUBE_URL}")
        part.rename(zip_path)
    rel_dst.mkdir(parents=True, exist_ok=True)
    found = set()
    with zipfile.ZipFile(zip_path) as z:
        for info in z.infolist():
            p = Path(info.filename)
            if p.name in NEEDED and "relations" in p.parts:
                with z.open(info) as src, open(rel_dst / p.name, "wb") as dst:
                    shutil.copyfileobj(src, dst)
                found.add(p.name)
    missing = NEEDED - found
    assert not missing, f"not found in zip: {missing}"
sh(f"ls -la {rel_dst}")

## 5. Link `tmp/` to persistent storage

In [ ]:
persist_tmp = PERSIST / "tmp"
persist_tmp.mkdir(exist_ok=True)
link = REPO_DIR / "tmp"
if link.is_symlink():
    link.unlink()
elif link.exists():
    shutil.rmtree(link)
link.symlink_to(persist_tmp, target_is_directory=True)
print(link, "->", link.resolve())

## 6. Run config (optional epoch override)

In [ ]:
cfg = json.load(open(CONFIG))
if EPOCHS_EMBED:
    cfg["TRAIN_EMBEDS"]["epochs"] = EPOCHS_EMBED
    cfg["TRAIN_EMBEDS"]["min_epochs"] = max(EPOCHS_EMBED - 1, 0)
if EPOCHS_AGENT:
    for k in ("TRAIN_AGENT", "TEST_AGENT"):
        cfg[k]["epochs"] = EPOCHS_AGENT
    cfg["TRAIN_AGENT"]["min_epochs"] = max(EPOCHS_AGENT - 1, 0)

RUN_CONFIG = "/content/mooc_colab.json"
json.dump(cfg, open(RUN_CONFIG, "w"), indent=2)

tmp_dir = cfg["PREPROCESS"]["tmp_dir"]
# Markers are written only after a step succeeds and include the epoch count,
# so a smoke run (2 epochs) never makes a full run (e.g. 30 epochs) skip.
MARK_DATASET = f"{tmp_dir}/.done_make_dataset"
MARK_TRANSE = f"{tmp_dir}/.done_train_transe_e{cfg['TRAIN_EMBEDS']['epochs']}"
MARK_AGENT = f"{tmp_dir}/.done_train_agent_e{cfg['TRAIN_AGENT']['epochs']}"
print("config:", RUN_CONFIG, "| tmp_dir:", tmp_dir)
print("markers:", MARK_DATASET, MARK_TRANSE, MARK_AGENT, sep="\n  ")


def step(title, script, marker=None):
    if marker and (REPO_DIR / marker).exists():
        print(f"[skip] {title}: {marker} exists")
        return
    print(f"=== {title} ===")
    sh(f"{PY} {script} --config {RUN_CONFIG}", cwd=REPO_DIR)
    if marker:
        (REPO_DIR / marker).touch()

## 7. Pipeline

In [ ]:
# Always runs (fast, idempotent). Later steps read data/mooc/processed_files.
print("=== preprocess ===")
sh(f"{PY} src/UPGPR/preprocess_mooc.py", cwd=REPO_DIR)

In [ ]:
step("make_dataset", "src/UPGPR/make_dataset.py", marker=MARK_DATASET)

In [ ]:
step("train_transe", "src/UPGPR/train_transe_model.py", marker=MARK_TRANSE)

In [ ]:
# Not resumable: if the runtime dies mid-training, this step starts over.
step("train_agent", "src/UPGPR/train_agent.py", marker=MARK_AGENT)

In [ ]:
step("test_agent", "src/UPGPR/test_agent.py")

## 8. Results

In [ ]:
res = Path(tmp_dir) / cfg["TEST_AGENT"]["result_file_dir"] / cfg["TEST_AGENT"]["result_file_name"]
print(res)
print(json.dumps(json.load(open(res)), indent=2))

## Optional: archive outputs (when Drive is not mounted)

Creates a zip you can download from the Colab file browser before the runtime ends.

In [ ]:
archive = shutil.make_archive("/content/upgpr_mooc_tmp", "zip", root_dir=str(Path(tmp_dir).resolve()))
print(archive, f"{os.path.getsize(archive) / 1e6:.1f} MB")